# Verify and visualize the graph $G_{r,k,h}$

Edit the parameters in the next cell, then run all cells. The notebook constructs the graph from the document, computes the signless Laplacian matrix $Q$, the first four columns of the $Q$-walk matrix

$$[\mathbf 1, Q\mathbf 1, Q^2\mathbf 1, Q^3\mathbf 1],$$

prints their rank, checks the defining equation, and reports the number of $Q$-main eigenvalues.

The notebook also writes matrix files to disk. For large graphs, $Q$ is exported as sparse coordinate data and Matrix Market format, and $[\mathbf 1,Q\mathbf 1,Q^2\mathbf 1,Q^3\mathbf 1]$ is exported as a four-column CSV.

The visualization section separates three different needs. The main large-graph picture is now a paper-style compressed drawing: the ladder core, the recursive branch $R_h$, and the local expansion $F_r$ are drawn as representative modules with multiplicity labels, so the structure stays readable even when the exact graph has thousands of vertices. For small graphs, the notebook can also draw the exact graph with a tidy rooted-tree layout. DOT/GEXF files are exported for Graphviz or Gephi exploration, but force-directed SVG rendering is disabled by default because it is slow and usually less suitable for a paper figure. For paper-style understanding it also draws a compressed schematic of the whole family $\{G_{r,k,h}:h\ge 0\}$ with $r,k$ fixed. Long explanations are printed outside the figures so the labels do not collide with the graph.


In [1]:
# Smoke test: run this cell first in VSCode.
from sage.env import SAGE_VERSION
import time
print(f"Smoke test: Sage kernel is running. Sage version={SAGE_VERSION}; time={time.strftime('%H:%M:%S')}")


Smoke test: Sage kernel is running. Sage version=10.7; time=16:53:33


In [ ]:
# Choose parameters here.
# r must be >= 2, k and h must be >= 0.
r = 2
k = 0
h = 0

# Matrix display controls. Q and W4 are always computed.
# For large graphs, set SHOW_FULL_MATRICES = False to avoid a very large notebook output.
SHOW_FULL_MATRICES = False
EXPORT_MATRIX_FILES = True
MATRIX_EXPORT_FORMATS = ("mtx", "csv", "json", "md")


# Safety controls. The graph size grows very fast with h.
# Set MAX_BUILD_VERTICES = None only if you really want to attempt a huge construction.
MAX_BUILD_VERTICES = 50000
MAX_FULL_MATRIX_DISPLAY_VERTICES = 400

# Visualization controls.
# The default notebook output avoids vertex labels and force-directed Sage plots,
# because those quickly obscure the graph structure in papers.
DRAW_DIRECT_SAGE_PLOT = False
MAX_SAGE_DRAW_VERTICES = 120
DRAW_STRUCTURAL_LAYOUT = True
MAX_STRUCTURAL_LAYOUT_VERTICES = 1200
STRUCTURAL_LAYOUT_FORMATS = ("png", "pdf")
DISPLAY_STRUCTURAL_IMAGE_IN_NOTEBOOK = True
CHECK_STRUCTURAL_EDGE_CROSSINGS = True
MAX_CROSSING_CHECK_EDGES = 2500
DRAW_COMPRESSED_MEMBER_SCHEMATIC = True
COMPRESSED_MEMBER_SCHEMATIC_FORMATS = ("png", "pdf")
DRAW_PLANAR_LAYOUT = False
MAX_PLANAR_LAYOUT_VERTICES = 5000
PLANAR_LAYOUT_FORMATS = ("png", "pdf")
DISPLAY_PLANAR_IMAGE_IN_NOTEBOOK = False
USE_GRAPHVIZ_SFDP = False
GRAPHVIZ_FULL_ENGINE = "sfdp"
GRAPHVIZ_SKELETON_ENGINE = "sfdp"
DISPLAY_SVG_IN_NOTEBOOK = False
MAX_GRAPHVIZ_RENDER_VERTICES = 800
EXPORT_DOT = True
EXPORT_GEXF = True
DRAW_SKELETON_VIEW = True

# Paper-style schematic for the whole infinite family with r,k fixed and h varying.
DRAW_INFINITE_FAMILY_SCHEMATIC = True
FAMILY_SCHEMATIC_FORMATS = ("png", "pdf")

# Optional: save a Sage plot image for small graphs.
SAVE_GRAPH_IMAGE = False
GRAPH_IMAGE_PATH = f"G_r{r}_k{k}_h{h}.png"

print(f"Parameter cell executed: r={r}, k={k}, h={h}; SHOW_FULL_MATRICES={SHOW_FULL_MATRICES}")


In [ ]:
from sage.all import Graph, ZZ, vector, matrix, diagonal_matrix, show
from pathlib import Path
import shutil
import subprocess
import csv
import json
from xml.sax.saxutils import escape as xml_escape

try:
    from IPython.display import SVG, Image, Markdown, display
except Exception:
    SVG = None
    Image = None
    Markdown = None
    display = None


def display_note(title, body=""):
    """Display a short Markdown note in notebooks, with a plain-text fallback."""
    body = str(body).strip()
    if Markdown is not None and display is not None:
        display(Markdown(f"### {title}\n{body}" if body else f"### {title}"))
    else:
        print("\n" + title)
        if body:
            print(body)


def _add_vertex(adj, signs, kinds, sign, kind):
    v = len(adj)
    adj[v] = set()
    signs[v] = sign
    kinds[v] = kind
    return v


def _add_edge(adj, u, v):
    if u == v:
        raise ValueError("loop attempted")
    if v in adj[u]:
        raise ValueError(f"multiple edge attempted: {u}, {v}")
    adj[u].add(v)
    adj[v].add(u)


def _delete_leaf(adj, signs, kinds, leaf):
    if len(adj[leaf]) != 1:
        raise ValueError("chosen vertex is not a leaf")
    parent = next(iter(adj[leaf]))
    adj[parent].remove(leaf)
    del adj[leaf]
    del signs[leaf]
    del kinds[leaf]
    return parent


def _relabel_compact(adj, signs, kinds):
    old_vertices = sorted(adj)
    mp = {v: i for i, v in enumerate(old_vertices)}
    new_adj = {mp[v]: set(mp[u] for u in adj[v]) for v in old_vertices}
    new_signs = {mp[v]: signs[v] for v in old_vertices}
    new_kinds = {mp[v]: kinds[v] for v in old_vertices}
    return new_adj, new_signs, new_kinds


def make_ladder_core(k):
    """Return the k-cyclic ladder core C_k with a fixed bipartition sign."""
    if k < 0:
        raise ValueError("k must be >= 0")
    adj, signs, kinds = {}, {}, {}
    u, v = [], []
    for i in range(k + 1):
        u.append(_add_vertex(adj, signs, kinds, 1 if i % 2 == 0 else -1, "core_u"))
    for i in range(k + 1):
        v.append(_add_vertex(adj, signs, kinds, -1 if i % 2 == 0 else 1, "core_v"))
    for i in range(k):
        _add_edge(adj, u[i], u[i + 1])
        _add_edge(adj, v[i], v[i + 1])
    for i in range(k + 1):
        _add_edge(adj, u[i], v[i])
    return adj, signs, kinds


def make_skeleton(r, k, h):
    """Construct the skeleton S_{r,k,h}. Signs: W=+1, B=-1."""
    if r < 2:
        raise ValueError("r must be >= 2")
    if k < 0 or h < 0:
        raise ValueError("k and h must be >= 0")

    X = r**2
    y = r + 1
    adj, signs, kinds = make_ladder_core(k)

    core_vertices = sorted(adj)

    # Complete old W-core vertices to degree X using new B vertices.
    for w in core_vertices:
        if signs[w] != 1:
            continue
        need = X - len(adj[w])
        if need < 0:
            raise ValueError("core W degree exceeds X")
        for _ in range(need):
            b = _add_vertex(adj, signs, kinds, -1, "fill_B_from_W")
            _add_edge(adj, w, b)
            for __ in range(y - 1):
                leaf = _add_vertex(adj, signs, kinds, 1, "W_leaf_from_fill_B")
                _add_edge(adj, b, leaf)

    # Complete old B-core vertices to degree y using W leaves.
    for b in core_vertices:
        if signs[b] != -1:
            continue
        need = y - len(adj[b])
        if need < 0:
            raise ValueError("core B degree exceeds y")
        for _ in range(need):
            leaf = _add_vertex(adj, signs, kinds, 1, "W_leaf_from_core_B")
            _add_edge(adj, b, leaf)

    # Replace one W leaf by the rooted branch R_h.
    leaves = [v for v in adj if signs[v] == 1 and len(adj[v]) == 1]
    if not leaves:
        raise RuntimeError("no W leaf available for replacement")
    parent = _delete_leaf(adj, signs, kinds, leaves[0])
    adj, signs, kinds = _relabel_compact(adj, signs, kinds)
    parent = sorted([v for v in adj if signs[v] == -1 and kinds[v] in ("fill_B_from_W", "core_v", "core_u", "core_B")], key=lambda z: len(adj[z]))[0] if parent not in adj else parent

    # The deletion compacts labels, so recover the unique B vertex whose degree dropped below y.
    dropped = [v for v in adj if signs[v] == -1 and len(adj[v]) == y - 1]
    if len(dropped) != 1:
        raise RuntimeError("could not identify replacement parent after deleting the leaf")
    parent = dropped[0]

    def add_R(depth, external_parent):
        root = _add_vertex(adj, signs, kinds, 1, f"R_{depth}_W")
        _add_edge(adj, external_parent, root)
        if depth >= 1:
            for _ in range(X - 1):
                b = _add_vertex(adj, signs, kinds, -1, f"R_{depth}_B")
                _add_edge(adj, root, b)
                for __ in range(y - 1):
                    add_R(depth - 1, b)
        return root

    add_R(h, parent)
    return adj, signs, kinds


def extend_F_r(skel_adj, skel_signs, skel_kinds, r):
    """Construct G_{r,k,h}=F_r(S_{r,k,h})."""
    X = r**2
    x = r**2 + 1
    y = r + 1
    z = r**2 + r + 1

    adj, signs, kinds = {}, {}, {}
    mp = {}
    for v in sorted(skel_adj):
        kind = "skeleton_W" if skel_signs[v] == 1 else "skeleton_B"
        mp[v] = _add_vertex(adj, signs, kinds, skel_signs[v], kind)
    for u in skel_adj:
        for v in skel_adj[u]:
            if u < v:
                _add_edge(adj, mp[u], mp[v])

    for w in sorted(skel_adj):
        if skel_signs[w] != 1:
            continue
        gw = mp[w]
        dS = len(skel_adj[w])
        if dS == X:
            q = _add_vertex(adj, signs, kinds, -1, "support")
            _add_edge(adj, gw, q)
            for _ in range(z - 1):
                leaf = _add_vertex(adj, signs, kinds, 1, "support_leaf")
                _add_edge(adj, q, leaf)
        elif dS == 1:
            for _ in range(r):
                q = _add_vertex(adj, signs, kinds, -1, "support")
                _add_edge(adj, gw, q)
                for __ in range(z - 1):
                    leaf = _add_vertex(adj, signs, kinds, 1, "support_leaf")
                    _add_edge(adj, q, leaf)
            for _ in range(r * (r - 1)):
                leaf = _add_vertex(adj, signs, kinds, -1, "direct_negative_leaf")
                _add_edge(adj, gw, leaf)
        else:
            raise ValueError(f"bad W skeleton degree {dS}; expected 1 or {X}")

    return adj, signs, kinds


def graph_from_adj(adj):
    edges = []
    for u in adj:
        for v in adj[u]:
            if u < v:
                edges.append((u, v))
    return Graph(edges, loops=False, multiedges=False)


def signless_laplacian_matrix(G, vertices=None):
    if vertices is None:
        vertices = G.vertices(sort=True)
    # Use sparse matrices so large graphs do not allocate a dense n by n matrix.
    A = G.adjacency_matrix(vertices=vertices, sparse=True)
    degs = [G.degree(v) for v in vertices]
    D = diagonal_matrix(ZZ, degs, sparse=True)
    return A + D


def walk_matrix_first_four(Q):
    n = Q.nrows()
    one = vector(ZZ, [1] * n)
    cols = [one]
    for _ in range(3):
        cols.append(Q * cols[-1])
    return matrix(ZZ, n, 4, lambda i, j: cols[j][i])




def _ladder_core_degree_sums(k):
    """Return degree lists for the + and - sides of C_k without building the full graph."""
    plus_degrees = []
    minus_degrees = []
    for i in range(k + 1):
        deg_u = (1 if i > 0 else 0) + (1 if i < k else 0) + 1
        deg_v = (1 if i > 0 else 0) + (1 if i < k else 0) + 1
        if i % 2 == 0:
            plus_degrees.append(deg_u)
            minus_degrees.append(deg_v)
        else:
            minus_degrees.append(deg_u)
            plus_degrees.append(deg_v)
    return plus_degrees, minus_degrees


def estimate_G_r_k_h_size(r, k, h):
    """Exact vertex/edge count estimates for S_{r,k,h} and G_{r,k,h}, without constructing them."""
    if r < 2 or k < 0 or h < 0:
        raise ValueError("need r>=2 and k,h>=0")
    X = r**2
    y = r + 1
    z = r**2 + r + 1
    plus_degrees, minus_degrees = _ladder_core_degree_sums(k)

    core_W = len(plus_degrees)
    core_B = len(minus_degrees)
    fill_B_from_W = sum(X - d for d in plus_degrees)
    W_leaves_from_fill_B = fill_B_from_W * (y - 1)
    W_leaves_from_core_B = sum(y - d for d in minus_degrees)
    base_W1 = W_leaves_from_fill_B + W_leaves_from_core_B
    base_WX = core_W
    base_B = core_B + fill_B_from_W

    # Counts inside R_h by skeleton type.
    R_WX, R_W1, R_B = 0, 1, 0
    branch_factor = (X - 1) * (y - 1)
    for _ in range(h):
        R_WX, R_W1, R_B = (
            1 + branch_factor * R_WX,
            branch_factor * R_W1,
            (X - 1) + branch_factor * R_B,
        )

    S_WX = base_WX + R_WX
    S_W1 = base_W1 - 1 + R_W1
    S_B = base_B + R_B
    S_vertices = S_WX + S_W1 + S_B
    S_edges = S_vertices - 1 + k

    add_for_WX = z
    add_for_W1 = r * z + r * (r - 1)
    G_vertices = S_vertices + S_WX * add_for_WX + S_W1 * add_for_W1
    G_edges = G_vertices - 1 + k
    return {
        "S_vertices": S_vertices,
        "S_edges": S_edges,
        "S_WX": S_WX,
        "S_W1": S_W1,
        "S_B": S_B,
        "G_vertices": G_vertices,
        "G_edges": G_edges,
        "omega": k,
    }


def build_G_r_k_h(r, k, h):
    skel_adj, skel_signs, skel_kinds = make_skeleton(r, k, h)
    adj, signs, kinds = extend_F_r(skel_adj, skel_signs, skel_kinds, r)
    G = graph_from_adj(adj)
    return G, adj, signs, kinds, skel_adj, skel_signs, skel_kinds


def verify_basic_identities(G, signs, r):
    a = r**2 + r + 3
    b = r + 2
    t = r
    vertices = G.vertices(sort=True)
    Q = signless_laplacian_matrix(G, vertices=vertices)
    n = G.num_verts()
    one = vector(ZZ, [1] * n)
    sigma = vector(ZZ, [signs[v] for v in vertices])
    q1 = Q * one
    q2 = Q * q1
    q3 = Q * q2
    residual = q2 - a * q1 + b * one - t * sigma
    q_sigma = Q * sigma
    recurrence = q3 - a * q2 + b * q1
    return Q, residual, q_sigma, recurrence




def matrix_nonzero_entries(M):
    """Return sorted nonzero entries (i,j,value) using zero-based row/column indices."""
    try:
        data = M.dict()
        return sorted((int(i), int(j), int(v)) for (i, j), v in data.items() if v != 0)
    except Exception:
        entries = []
        for i in range(M.nrows()):
            for j in range(M.ncols()):
                v = M[i, j]
                if v != 0:
                    entries.append((int(i), int(j), int(v)))
        return entries


def json_ready(obj):
    """Convert Sage numeric containers into ordinary JSON-serializable Python objects."""
    if isinstance(obj, dict):
        return {str(k): json_ready(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_ready(v) for v in obj]
    if isinstance(obj, bool) or obj is None or isinstance(obj, (str, float)):
        return obj
    try:
        return int(obj)
    except Exception:
        return str(obj)


def export_matrix_files(Q, W4, vertices, out_dir, metadata):
    """Export Q and the first four Q-walk columns in AI-readable file formats."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    q_entries = matrix_nonzero_entries(Q)
    n = int(Q.nrows())
    wcols = ["vertex", "one", "Q_one", "Q2_one", "Q3_one"]

    paths = {}
    if "csv" in MATRIX_EXPORT_FORMATS:
        q_csv = out_dir / "Q_sparse_entries.csv"
        with q_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, lineterminator="\n")
            writer.writerow(["row_index_0based", "col_index_0based", "row_vertex", "col_vertex", "value"])
            for i, j, val in q_entries:
                writer.writerow([i, j, vertices[i], vertices[j], val])
        paths["Q_sparse_entries_csv"] = q_csv

        w_csv = out_dir / "Q_walk_first_four_columns.csv"
        with w_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, lineterminator="\n")
            writer.writerow(wcols)
            for i, vtx in enumerate(vertices):
                writer.writerow([vtx, int(W4[i, 0]), int(W4[i, 1]), int(W4[i, 2]), int(W4[i, 3])])
        paths["W4_csv"] = w_csv

        vertex_csv = out_dir / "vertex_order.csv"
        with vertex_csv.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, lineterminator="\n")
            writer.writerow(["row_index_0based", "vertex_label"])
            for i, vtx in enumerate(vertices):
                writer.writerow([i, vtx])
        paths["vertex_order_csv"] = vertex_csv

    if "mtx" in MATRIX_EXPORT_FORMATS:
        q_mtx = out_dir / "Q_signless_laplacian.mtx"
        with q_mtx.open("w", encoding="utf-8") as f:
            f.write("%%MatrixMarket matrix coordinate integer general\n")
            f.write("% Signless Laplacian Q=A+D. Indices below are 1-based Matrix Market indices.\n")
            f.write(f"{n} {n} {len(q_entries)}\n")
            for i, j, val in q_entries:
                f.write(f"{i + 1} {j + 1} {val}\n")
        paths["Q_mtx"] = q_mtx

    meta = dict(metadata)
    meta.update({
        "matrix_row_order": "Rows and columns use vertex_order.csv; CSV indices are 0-based, Matrix Market indices are 1-based.",
        "Q_shape": [int(Q.nrows()), int(Q.ncols())],
        "Q_nonzeros": len(q_entries),
        "W4_shape": [int(W4.nrows()), int(W4.ncols())],
        "W4_columns": wcols[1:],
        "files": {key: str(path) for key, path in paths.items()},
    })

    if "json" in MATRIX_EXPORT_FORMATS:
        meta_json = out_dir / "matrix_metadata.json"
        meta_json.write_text(json.dumps(json_ready(meta), ensure_ascii=False, indent=int(2)) + "\n", encoding="utf-8")
        paths["metadata_json"] = meta_json

    if "md" in MATRIX_EXPORT_FORMATS:
        readme = out_dir / "README_matrices.md"
        readme.write_text(
            "# Matrix artifacts for G_{r,k,h}\n\n"
            f"Parameters: `r={metadata.get('r')}`, `k={metadata.get('k')}`, `h={metadata.get('h')}`.\n\n"
            f"`Q` has shape `{Q.nrows()} x {Q.ncols()}` and `{len(q_entries)}` nonzero entries.\n"
            f"`W4=[1,Q1,Q^2 1,Q^3 1]` has shape `{W4.nrows()} x {W4.ncols()}`.\n\n"
            "Files:\n"
            "- `Q_signless_laplacian.mtx`: Matrix Market sparse coordinate representation of Q.\n"
            "- `Q_sparse_entries.csv`: sparse coordinate table with row/column vertex labels.\n"
            "- `Q_walk_first_four_columns.csv`: the four Q-walk columns, one vertex per row.\n"
            "- `vertex_order.csv`: row/column order used by Q and W4.\n"
            "- `matrix_metadata.json`: parameters, dimensions, recurrence, and Q-main eigenvalue summary.\n\n"
            "A reader can reconstruct Q from either the Matrix Market file or the sparse CSV using `vertex_order.csv`.\n",
            encoding="utf-8",
        )
        paths["readme_md"] = readme

    print(f"Matrix artifact directory: {out_dir.resolve()}")
    for label, path in paths.items():
        print(f"  {label}: {path}")
    return paths


def write_dot(adj, signs, kinds, path, graph_name="G", label_vertices=False, node_width=0.070, edge_width=0.70):
    """Write an undirected DOT file with paper-style black/white bipartition styling."""
    path = Path(path)
    with path.open("w", encoding="utf-8") as f:
        f.write(f'graph "{graph_name}" {{\n')
        f.write('  graph [overlap=false, splines=true, sep="+6", K=1.15, outputorder=edgesfirst, bgcolor="white", pad=0.04, margin=0, forcelabels=false];\n')
        f.write(f'  node [shape=circle, style=filled, fixedsize=true, width={float(node_width):.3f}, height={float(node_width):.3f}, penwidth=0.85, label="", fontname="Times New Roman", fontsize=8];\n')
        f.write(f'  edge [color="#111111", penwidth={float(edge_width):.2f}];\n')
        for v in sorted(adj):
            fill = "white" if signs[v] == 1 else "black"
            font = "black" if signs[v] == 1 else "white"
            tooltip = xml_escape(f"v={v}, sign={signs[v]}, degree={len(adj[v])}, kind={kinds.get(v, '')}")
            xlabel = f', xlabel="{v}"' if label_vertices else ""
            f.write(
                f'  "{v}" [fillcolor="{fill}", color="black", fontcolor="{font}", tooltip="{tooltip}"{xlabel}];\n'
            )
        for u in sorted(adj):
            for v in sorted(adj[u]):
                if u < v:
                    f.write(f'  "{u}" -- "{v}";\n')
        f.write('}\n')
    return path


def write_gexf(adj, signs, kinds, path, graph_label="G"):
    """Write a GEXF file for Gephi, including sign/kind/degree attributes and colors."""
    path = Path(path)
    rgb = {1: (255, 255, 255), -1: (0, 0, 0)}
    with path.open("w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        f.write('<gexf xmlns="http://gexf.net/1.3" xmlns:viz="http://gexf.net/1.3/viz" version="1.3">\n')
        f.write(f'  <meta><creator>SageMath notebook</creator><description>{xml_escape(graph_label)}</description></meta>\n')
        f.write('  <graph mode="static" defaultedgetype="undirected">\n')
        f.write('    <attributes class="node">\n')
        f.write('      <attribute id="sign" title="sign" type="integer" />\n')
        f.write('      <attribute id="kind" title="kind" type="string" />\n')
        f.write('      <attribute id="degree" title="degree" type="integer" />\n')
        f.write('    </attributes>\n')
        f.write('    <nodes>\n')
        for v in sorted(adj):
            r0, g0, b0 = rgb[signs[v]]
            kind = xml_escape(str(kinds.get(v, "")))
            f.write(f'      <node id="{v}" label="{v}">\n')
            f.write('        <attvalues>\n')
            f.write(f'          <attvalue for="sign" value="{signs[v]}" />\n')
            f.write(f'          <attvalue for="kind" value="{kind}" />\n')
            f.write(f'          <attvalue for="degree" value="{len(adj[v])}" />\n')
            f.write('        </attvalues>\n')
            f.write(f'        <viz:color r="{r0}" g="{g0}" b="{b0}" a="1.0" />\n')
            f.write('      </node>\n')
        f.write('    </nodes>\n')
        f.write('    <edges>\n')
        eid = 0
        for u in sorted(adj):
            for v in sorted(adj[u]):
                if u < v:
                    f.write(f'      <edge id="{eid}" source="{u}" target="{v}" />\n')
                    eid += 1
        f.write('    </edges>\n')
        f.write('  </graph>\n')
        f.write('</gexf>\n')
    return path


def render_graphviz(dot_path, svg_path, engine="sfdp"):
    """Render DOT to SVG using a Graphviz engine. Returns True if rendering succeeded."""
    engine = str(engine)
    executable = shutil.which(engine)
    if not executable:
        print(f"Graphviz {engine!r} was not found on PATH. DOT/GEXF exports are still available.")
        return False
    dot_path = Path(dot_path)
    svg_path = Path(svg_path)
    cmd = [executable, "-Tsvg", str(dot_path), "-o", str(svg_path)]
    print("Running Graphviz:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    return True


def display_svg_if_possible(svg_path):
    svg_path = Path(svg_path)
    if SVG is not None and display is not None and svg_path.exists():
        display(SVG(filename=str(svg_path)))
    elif svg_path.exists():
        print(f"SVG written to {svg_path}")


def export_visualization_bundle(
    name,
    adj,
    signs,
    kinds,
    out_dir,
    description="",
    engine="sfdp",
    render_vertex_limit=20000,
    label_vertices=False,
    node_width=0.070,
    edge_width=0.70,
    display_svg=True,
):
    """Export DOT/GEXF and optionally render SVG with Graphviz."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    dot_path = out_dir / f"{name}.dot"
    gexf_path = out_dir / f"{name}.gexf"
    svg_path = out_dir / f"{name}_{engine}.svg"

    if description:
        display_note(name, description)

    if EXPORT_DOT:
        write_dot(adj, signs, kinds, dot_path, graph_name=name, label_vertices=label_vertices, node_width=node_width, edge_width=edge_width)
        print(f"DOT file: {dot_path}")
    if EXPORT_GEXF:
        write_gexf(adj, signs, kinds, gexf_path, graph_label=name)
        print(f"GEXF file for Gephi: {gexf_path}")

    if USE_GRAPHVIZ_SFDP and EXPORT_DOT:
        if len(adj) <= render_vertex_limit:
            if render_graphviz(dot_path, svg_path, engine=engine):
                print(f"SVG rendered by Graphviz {engine}: {svg_path}")
                if display_svg:
                    display_svg_if_possible(svg_path)
        else:
            print(f"Skipped Graphviz rendering for {name}: {len(adj)} vertices exceeds limit {render_vertex_limit}.")
            print("Open the DOT/GEXF files in Graphviz/Gephi, or raise MAX_GRAPHVIZ_RENDER_VERTICES.")
    return dot_path, gexf_path, svg_path


def ladder_core_data(k, adj):
    """Return the intended ladder-core vertices and edges for G_{r,k,h} or S_{r,k,h}."""
    if k < 0:
        return [], [], []
    u_vertices = list(range(k + 1))
    v_vertices = list(range(k + 1, 2 * k + 2))
    core_vertices = u_vertices + v_vertices
    if any(v not in adj for v in core_vertices):
        return [], [], []
    core_edges = set()
    for i in range(k):
        core_edges.add(tuple(sorted((u_vertices[i], u_vertices[i + 1]))))
        core_edges.add(tuple(sorted((v_vertices[i], v_vertices[i + 1]))))
    for i in range(k + 1):
        core_edges.add(tuple(sorted((u_vertices[i], v_vertices[i]))))
    return u_vertices, v_vertices, core_edges


def rooted_component_width(adj, node, parent, core_set, memo):
    key = (node, parent)
    if key in memo:
        return memo[key]
    children = [w for w in sorted(adj[node]) if w != parent and w not in core_set]
    if not children:
        width = 1.0
    else:
        width = max(1.0, sum(rooted_component_width(adj, w, node, core_set, memo) for w in children))
    memo[key] = width
    return width


def assign_rooted_component_positions(adj, node, parent, core_set, pos, x_left, x_right, y0, orient, depth, level_gap, memo):
    x_mid = (float(x_left) + float(x_right)) / 2.0
    y = float(y0) + float(orient) * float(depth) * float(level_gap)
    pos[node] = (x_mid, y)
    children = [w for w in sorted(adj[node]) if w != parent and w not in core_set]
    if not children:
        return
    total = sum(rooted_component_width(adj, w, node, core_set, memo) for w in children)
    cursor = float(x_left)
    span = float(x_right) - float(x_left)
    for child in children:
        child_width = rooted_component_width(adj, child, node, core_set, memo)
        child_span = span * child_width / total if total else span / len(children)
        assign_rooted_component_positions(
            adj, child, node, core_set, pos, cursor, cursor + child_span, y0, orient, depth + 1, level_gap, memo
        )
        cursor += child_span


def structural_layout_positions(adj, signs, k, leaf_gap=0.24, level_gap=0.62, column_gap=3.25, core_gap=1.18):
    """Structure-aware radial layout for this constructed graph family.

    The ladder core is kept compact.  Each component attached to the core is a
    rooted tree and is drawn outward in its own angular sector.  This is closer
    to the way graph-theory papers draw unicyclic/k-cyclic graphs: a visible
    core with branches growing away from it.
    """
    import math

    u_vertices, v_vertices, core_edges = ladder_core_data(k, adj)
    if not u_vertices:
        raise ValueError("Could not identify the ladder core vertices for the structural layout.")

    core_set = set(u_vertices + v_vertices)
    memo = {}

    def branch_units(root):
        units = []
        for nbr in sorted(adj[root]):
            if nbr in core_set:
                continue
            units.append((nbr, rooted_component_width(adj, nbr, root, core_set, memo)))
        return units

    # Compact ladder core.
    pos = {}
    if k == 0:
        pos[u_vertices[0]] = (0.0, core_gap / 2.0)
        pos[v_vertices[0]] = (0.0, -core_gap / 2.0)
    else:
        x0 = -0.5 * column_gap * k
        for i, u in enumerate(u_vertices):
            pos[u] = (x0 + column_gap * i, core_gap / 2.0)
        for i, v in enumerate(v_vertices):
            pos[v] = (x0 + column_gap * i, -core_gap / 2.0)

    def subtree_depth(node, parent, core_set, memo_depth):
        key = (node, parent)
        if key in memo_depth:
            return memo_depth[key]
        children = [w for w in sorted(adj[node]) if w != parent and w not in core_set]
        d = 1 if not children else 1 + max(subtree_depth(w, node, core_set, memo_depth) for w in children)
        memo_depth[key] = d
        return d

    depth_memo = {}

    def place_child_sector(parent, child, center_angle, angle_span, step):
        px, py = pos[parent]
        cx = px + step * math.cos(center_angle)
        cy = py + step * math.sin(center_angle)
        pos[child] = (cx, cy)
        place_radial_children(child, parent, center_angle, angle_span, step)

    def place_radial_children(node, parent, center_angle, angle_span, step):
        children = [w for w in sorted(adj[node]) if w != parent and w not in core_set]
        if not children:
            return
        weights = [rooted_component_width(adj, w, node, core_set, memo) for w in children]
        total = sum(weights)
        # High-degree stars need wide fans; paths should stay narrow and readable.
        local_span = min(float(angle_span), max(0.42, 0.16 * len(children)))
        cursor = center_angle - local_span / 2.0
        for child, weight in zip(children, weights):
            child_span = local_span * weight / total if total else local_span / len(children)
            child_angle = cursor + child_span / 2.0
            place_child_sector(node, child, child_angle, max(0.18, child_span), step)
            cursor += child_span

    def outward_angle(root):
        x, y = pos[root]
        if k == 0:
            return math.pi / 2.0 if root in u_vertices else -math.pi / 2.0
        # Top-row branches grow upward, bottom-row branches downward.  A small
        # horizontal tilt keeps adjacent branches visually separated.
        idx = u_vertices.index(root) if root in u_vertices else v_vertices.index(root)
        mid = k / 2.0
        tilt = 0.20 * (idx - mid) / max(1.0, mid)
        return math.pi / 2.0 - tilt if root in u_vertices else -math.pi / 2.0 + tilt

    def root_wedge(root):
        if k == 0:
            return 2.30
        return max(0.85, min(1.55, 2.35 / (k + 1)))

    for root in u_vertices + v_vertices:
        units = branch_units(root)
        if not units:
            continue
        weights = [w for _, w in units]
        total = sum(weights)
        base_angle = outward_angle(root)
        span = root_wedge(root)
        cursor = base_angle - span / 2.0
        for child, weight in units:
            child_span = span * weight / total if total else span / len(units)
            child_angle = cursor + child_span / 2.0
            depth = subtree_depth(child, root, core_set, depth_memo)
            # Deep branches get a little more radial spacing; shallow stars stay compact.
            step = level_gap * (1.0 + min(0.35, 0.035 * depth))
            place_child_sector(root, child, child_angle, max(0.20, child_span), step)
            cursor += child_span

    missing = [v for v in adj if v not in pos]
    if missing:
        raise RuntimeError(f"Structural layout missed {len(missing)} vertices; first missing vertex is {missing[0]}")
    return pos, core_edges



def skeleton_blowup_layout_positions(adj, signs, kinds, k):
    """Draw full G_{r,k,h} as a readable blow-up of its skeleton.

    The full graph is obtained from the skeleton S_{r,k,h} by attaching small
    depth-two modules at the W-vertices.  For a paper-style figure, drawing the
    skeleton first and then expanding these local modules outward preserves the
    construction far better than a generic planar embedding.
    """
    import math

    skeleton_set = {v for v in adj if str(kinds.get(v, "")).startswith("skeleton_")}
    if not skeleton_set:
        return None

    skel_adj = {v: set(w for w in adj[v] if w in skeleton_set) for v in skeleton_set}
    skel_signs = {v: signs[v] for v in skeleton_set}
    skel_pos, skel_core_edges = structural_layout_positions(
        skel_adj,
        skel_signs,
        k,
        leaf_gap=0.34,
        level_gap=0.82,
        column_gap=3.65,
        core_gap=1.32,
    )

    # Estimate the radius needed for the F_r modules and scale the skeleton so
    # neighboring modules do not compete for the same drawing area.
    max_module_width = 0.0
    for w in skeleton_set:
        extras = [u for u in adj[w] if u not in skeleton_set]
        width = 0.0
        for u in extras:
            leaves = [x for x in adj[u] if x != w and x not in skeleton_set]
            if leaves:
                width += max(0.76, (len(leaves) - 1) * 0.18) + 0.42
            else:
                width += 0.32
        max_module_width = max(max_module_width, width)
    scale = max(2.45, 1.75 + 0.42 * max_module_width)

    pos = {v: (scale * float(x), scale * float(y)) for v, (x, y) in skel_pos.items()}
    core_edges = {tuple(sorted(e)) for e in skel_core_edges}

    def normalize(vec, fallback=(0.0, 1.0)):
        x, y = float(vec[0]), float(vec[1])
        n = math.hypot(x, y)
        if n <= 1e-9:
            return fallback
        return (x / n, y / n)

    def outward_direction(w):
        skel_neighbors = [u for u in adj[w] if u in skeleton_set]
        wx, wy = pos[w]
        if not skel_neighbors:
            return (0.0, 1.0)
        cx = sum(pos[u][0] for u in skel_neighbors) / len(skel_neighbors)
        cy = sum(pos[u][1] for u in skel_neighbors) / len(skel_neighbors)
        outward = normalize((wx - cx, wy - cy), fallback=(0.0, 1.0 if signs[w] == 1 else -1.0))
        # If the local barycenter points almost horizontally for a ladder-core
        # vertex, bias slightly away from the row so the attached module reads as
        # an external pendant decoration rather than part of the core edge.
        if abs(outward[1]) < 0.18:
            row_bias = 1.0 if wy >= 0 else -1.0
            outward = normalize((outward[0], outward[1] + 0.42 * row_bias), fallback=(0.0, row_bias))
        return outward

    leaf_spacing = 0.18
    module_gap = 0.42
    support_dist = 0.76
    leaf_dist = 0.64
    direct_dist = 1.15

    for w in sorted(skeleton_set):
        if signs[w] != 1:
            continue
        extras = [u for u in sorted(adj[w]) if u not in skeleton_set]
        if not extras:
            continue

        outward = outward_direction(w)
        tangent = (-outward[1], outward[0])
        modules = []
        for u in extras:
            leaves = sorted(x for x in adj[u] if x != w and x not in skeleton_set)
            if leaves:
                width = max(0.76, (len(leaves) - 1) * leaf_spacing)
                modules.append((u, leaves, width))
            else:
                modules.append((u, [], 0.30))
        total_width = sum(width for _, _, width in modules) + module_gap * max(0, len(modules) - 1)
        cursor = -total_width / 2.0
        wx, wy = pos[w]

        for u, leaves, width in modules:
            center = cursor + width / 2.0
            if leaves:
                ux = wx + support_dist * outward[0] + center * tangent[0]
                uy = wy + support_dist * outward[1] + center * tangent[1]
                pos[u] = (ux, uy)
                leaf_start = center - (len(leaves) - 1) * leaf_spacing / 2.0
                for j, leaf in enumerate(leaves):
                    off = leaf_start + j * leaf_spacing
                    lx = wx + (support_dist + leaf_dist) * outward[0] + off * tangent[0]
                    ly = wy + (support_dist + leaf_dist) * outward[1] + off * tangent[1]
                    pos[leaf] = (lx, ly)
            else:
                lx = wx + direct_dist * outward[0] + center * tangent[0]
                ly = wy + direct_dist * outward[1] + center * tangent[1]
                pos[u] = (lx, ly)
            cursor += width + module_gap

    missing = [v for v in adj if v not in pos]
    if missing:
        raise RuntimeError(f"Skeleton blow-up layout missed {len(missing)} vertices; first missing vertex is {missing[0]}")
    return pos, core_edges


def ladder_tidy_layout_positions(adj, signs, k, leaf_gap=0.145, sibling_gap=0.105, component_gap=0.22, level_gap=0.64, core_gap=1.45, min_column_gap=2.15):
    """Crossing-free, paper-style layout for the ladder-core construction.

    The drawing fixes the k-cyclic ladder core as the visible spine. Every
    component attached to a core vertex is a rooted tree, drawn outward with a
    tidy layered layout: children occupy disjoint horizontal intervals and a
    parent is centered over its children. This is the standard visual grammar of
    graph-theory figures for trees/unicyclic/k-cyclic graphs.
    """
    u_vertices, v_vertices, core_edges = ladder_core_data(k, adj)
    if not u_vertices:
        raise ValueError("Could not identify the ladder core vertices for the tidy layout.")

    core_set = set(u_vertices + v_vertices)
    width_memo = {}

    def sorted_children(node, parent):
        return [w for w in sorted(adj[node]) if w != parent and w not in core_set]

    def subtree_width(node, parent):
        key = (node, parent)
        if key in width_memo:
            return width_memo[key]
        children = sorted_children(node, parent)
        if not children:
            width = leaf_gap
        else:
            child_widths = [subtree_width(c, node) for c in children]
            width = max(leaf_gap, sum(child_widths) + sibling_gap * (len(children) - 1))
        width_memo[key] = width
        return width

    def forest_width(root):
        children = [w for w in sorted(adj[root]) if w not in core_set]
        if not children:
            return leaf_gap
        widths = [subtree_width(c, root) for c in children]
        return max(leaf_gap, sum(widths) + component_gap * (len(children) - 1))

    top_widths = [forest_width(u) for u in u_vertices]
    bottom_widths = [forest_width(v) for v in v_vertices]
    column_widths = [max(top_widths[i], bottom_widths[i], 0.6) for i in range(k + 1)]

    column_x = [0.0]
    for i in range(k):
        gap = max(min_column_gap, 0.5 * (column_widths[i] + column_widths[i + 1]) + 0.85)
        column_x.append(column_x[-1] + gap)
    center_shift = 0.5 * (column_x[0] + column_x[-1])
    column_x = [x - center_shift for x in column_x]

    pos = {}
    for i, u in enumerate(u_vertices):
        pos[u] = (column_x[i], core_gap / 2.0)
    for i, v in enumerate(v_vertices):
        pos[v] = (column_x[i], -core_gap / 2.0)

    def assign_subtree(node, parent, center_x, y, direction, depth=0):
        pos[node] = (float(center_x), float(y))
        children = sorted_children(node, parent)
        if not children:
            return
        child_widths = [subtree_width(c, node) for c in children]
        total = sum(child_widths) + sibling_gap * (len(children) - 1)
        cursor = center_x - total / 2.0
        # Slightly increase vertical distance down a long branch so that large
        # stars remain legible without making the whole figure explode.
        step = level_gap * (1.0 + min(0.22, 0.025 * depth))
        child_y = y + direction * step
        for child, width in zip(children, child_widths):
            child_center = cursor + width / 2.0
            assign_subtree(child, node, child_center, child_y, direction, depth + 1)
            cursor += width + sibling_gap

    def assign_forest(root, direction):
        children = [w for w in sorted(adj[root]) if w not in core_set]
        if not children:
            return
        widths = [subtree_width(c, root) for c in children]
        total = sum(widths) + component_gap * (len(children) - 1)
        root_x, root_y = pos[root]
        cursor = root_x - total / 2.0
        child_y = root_y + direction * level_gap
        for child, width in zip(children, widths):
            child_center = cursor + width / 2.0
            assign_subtree(child, root, child_center, child_y, direction, 1)
            cursor += width + component_gap

    for u in u_vertices:
        assign_forest(u, 1.0)
    for v in v_vertices:
        assign_forest(v, -1.0)

    missing = [v for v in adj if v not in pos]
    if missing:
        raise RuntimeError(f"Tidy ladder layout missed {len(missing)} vertices; first missing vertex is {missing[0]}")
    return pos, core_edges


def ladder_oriented_tidy_layout_positions(adj, signs, k, leaf_gap=0.135, sibling_gap=0.095, component_gap=0.20, level_gap=0.60, core_gap=1.42):
    """Compact crossing-free layout: ladder core with outward tidy branches.

    This keeps the exact vertices, but uses the paper convention that trees
    attached to a cycle/core grow outward from the core.  Each attached tree is
    still a tidy rooted tree in its own local coordinate frame, so the drawing
    remains readable and the crossing checker can certify the result.
    """
    import math

    u_vertices, v_vertices, core_edges = ladder_core_data(k, adj)
    if not u_vertices:
        raise ValueError("Could not identify the ladder core vertices for the oriented tidy layout.")

    core_set = set(u_vertices + v_vertices)
    width_memo = {}

    def sorted_children(node, parent):
        return [w for w in sorted(adj[node]) if w != parent and w not in core_set]

    def subtree_width(node, parent):
        key = (node, parent)
        if key in width_memo:
            return width_memo[key]
        children = sorted_children(node, parent)
        if not children:
            width = leaf_gap
        else:
            widths = [subtree_width(c, node) for c in children]
            width = max(leaf_gap, sum(widths) + sibling_gap * (len(children) - 1))
        width_memo[key] = width
        return width

    def forest_width(root):
        children = [w for w in sorted(adj[root]) if w not in core_set]
        if not children:
            return leaf_gap
        widths = [subtree_width(c, root) for c in children]
        return max(leaf_gap, sum(widths) + component_gap * (len(children) - 1))

    col_widths = [max(forest_width(u_vertices[i]), forest_width(v_vertices[i]), 0.8) for i in range(k + 1)]
    # Less conservative than the purely horizontal tidy layout: branches are
    # tilted away from the core, so adjacent columns do not need all their leaf
    # intervals laid side by side.
    column_x = [0.0]
    for i in range(k):
        gap = max(3.2, 1.95 + 0.20 * (col_widths[i] + col_widths[i + 1]))
        column_x.append(column_x[-1] + gap)
    center_shift = 0.5 * (column_x[0] + column_x[-1])
    column_x = [x - center_shift for x in column_x]

    pos = {}
    for i, u in enumerate(u_vertices):
        pos[u] = (column_x[i], core_gap / 2.0)
    for i, v in enumerate(v_vertices):
        pos[v] = (column_x[i], -core_gap / 2.0)

    def normalize(x, y):
        n = math.hypot(x, y)
        if n <= 1e-9:
            return (0.0, 1.0)
        return (x / n, y / n)

    def frame_for_core(root):
        if root in u_vertices:
            i = u_vertices.index(root)
            vertical = 1.0
        else:
            i = v_vertices.index(root)
            vertical = -1.0
        if k == 0:
            horiz = 0.0
        else:
            mid = k / 2.0
            horiz = (i - mid) / max(mid, 1.0)
        # End columns lean outward; middle columns mostly grow vertically.
        dx, dy = normalize(0.72 * horiz, vertical)
        tx, ty = -dy, dx
        return (dx, dy), (tx, ty)

    def to_global(root, lateral, depth):
        rx, ry = pos[root]
        (dx, dy), (tx, ty) = frame_for_core(root)
        return (rx + lateral * tx + depth * dx, ry + lateral * ty + depth * dy)

    def assign_subtree(root, node, parent, lateral, depth, generation=0):
        pos[node] = to_global(root, lateral, depth)
        children = sorted_children(node, parent)
        if not children:
            return
        widths = [subtree_width(c, node) for c in children]
        total = sum(widths) + sibling_gap * (len(children) - 1)
        cursor = lateral - total / 2.0
        next_depth = depth + level_gap * (1.0 + min(0.20, 0.02 * generation))
        for child, width in zip(children, widths):
            child_lateral = cursor + width / 2.0
            assign_subtree(root, child, node, child_lateral, next_depth, generation + 1)
            cursor += width + sibling_gap

    def assign_forest(root):
        children = [w for w in sorted(adj[root]) if w not in core_set]
        if not children:
            return
        widths = [subtree_width(c, root) for c in children]
        total = sum(widths) + component_gap * (len(children) - 1)
        cursor = -total / 2.0
        for child, width in zip(children, widths):
            child_lateral = cursor + width / 2.0
            assign_subtree(root, child, root, child_lateral, level_gap, 1)
            cursor += width + component_gap

    for root in u_vertices + v_vertices:
        assign_forest(root)

    missing = [v for v in adj if v not in pos]
    if missing:
        raise RuntimeError(f"Oriented tidy layout missed {len(missing)} vertices; first missing vertex is {missing[0]}")
    return pos, core_edges


def choose_structural_layout_positions(adj, signs, k, kinds=None):
    # Default notebook figure: a compact ladder-core drawing with outward tidy
    # branches.  This mirrors the way graph-theory papers draw a visible
    # cycle/core with rooted trees attached, while preserving exact vertices.
    return ladder_oriented_tidy_layout_positions(adj, signs, k)


def edge_list_from_adj(adj):
    return [(u, v) for u in sorted(adj) for v in sorted(adj[u]) if u < v]


def _orientation(p, q, r, eps=1e-9):
    val = (q[0] - p[0]) * (r[1] - p[1]) - (q[1] - p[1]) * (r[0] - p[0])
    if abs(val) <= eps:
        return 0
    return 1 if val > 0 else -1


def _proper_segment_crossing(p1, p2, q1, q2):
    # Count only proper interior crossings. Shared endpoints are handled outside.
    o1 = _orientation(p1, p2, q1)
    o2 = _orientation(p1, p2, q2)
    o3 = _orientation(q1, q2, p1)
    o4 = _orientation(q1, q2, p2)
    return o1 * o2 < 0 and o3 * o4 < 0


def count_edge_crossings(adj, pos, max_edges=2500):
    edges = edge_list_from_adj(adj)
    if len(edges) > max_edges:
        return None
    crossings = 0
    examples = []
    for i, (a, b) in enumerate(edges):
        p1, p2 = pos[a], pos[b]
        for c, d in edges[i + 1:]:
            if len({a, b, c, d}) < 4:
                continue
            q1, q2 = pos[c], pos[d]
            if _proper_segment_crossing(p1, p2, q1, q2):
                crossings += 1
                if len(examples) < 5:
                    examples.append(((a, b), (c, d)))
    return crossings, examples


def draw_structural_layout_image(name, adj, signs, k, out_dir, description="", formats=("png", "pdf"), display_image=True, kinds=None):
    """Draw the exact graph with a construction-aware, crossing-free tree expansion layout."""
    try:
        import matplotlib.pyplot as plt
    except Exception as exc:
        print(f"Structural layout skipped because Matplotlib is unavailable: {exc}")
        return []

    if len(adj) > MAX_STRUCTURAL_LAYOUT_VERTICES:
        print(f"Structural layout skipped for {name}: {len(adj)} vertices > MAX_STRUCTURAL_LAYOUT_VERTICES={MAX_STRUCTURAL_LAYOUT_VERTICES}.")
        return []

    if description:
        display_note(name + " structural layout", description)

    pos, core_edges = choose_structural_layout_positions(adj, signs, k, kinds=kinds)
    crossing_result = None
    if CHECK_STRUCTURAL_EDGE_CROSSINGS:
        crossing_result = count_edge_crossings(adj, pos, max_edges=MAX_CROSSING_CHECK_EDGES)
        if crossing_result is None:
            print(f"Structural layout crossing check skipped: more than {MAX_CROSSING_CHECK_EDGES} edges.")
        else:
            crossings, examples = crossing_result
            print(f"Structural layout proper edge crossings: {crossings}")
            if crossings:
                print(f"  first crossing examples: {examples}")

    xs = [float(pos[v][0]) for v in pos]
    ys = [float(pos[v][1]) for v in pos]
    width = max(xs) - min(xs) if xs else 1.0
    height = max(ys) - min(ys) if ys else 1.0
    fig_w = max(8.0, min(28.0, width * 0.42 + 2.0))
    fig_h = max(5.5, min(22.0, height * 0.62 + 1.5))
    fig, ax = plt.subplots(figsize=(float(fig_w), float(fig_h)))
    ax.set_aspect("equal")
    ax.axis("off")

    for u, v in edge_list_from_adj(adj):
        p, q = pos[u], pos[v]
        is_core = tuple(sorted((u, v))) in core_edges
        lw = 1.25 if is_core else 0.78
        ax.plot([p[0], q[0]], [p[1], q[1]], color="black", linewidth=float(lw), zorder=1)

    n = len(adj)
    node_size = 42 if n <= 180 else 28 if n <= 600 else 18
    white_nodes = [v for v in sorted(adj) if signs[v] == 1]
    black_nodes = [v for v in sorted(adj) if signs[v] == -1]
    ax.scatter([pos[v][0] for v in white_nodes], [pos[v][1] for v in white_nodes],
               s=float(node_size), facecolors="white", edgecolors="black", linewidths=float(0.85), zorder=3)
    ax.scatter([pos[v][0] for v in black_nodes], [pos[v][1] for v in black_nodes],
               s=float(node_size), facecolors="black", edgecolors="black", linewidths=float(0.85), zorder=3)

    margin_x = max(0.5, width * 0.035)
    margin_y = max(0.5, height * 0.055)
    ax.set_xlim(min(xs) - margin_x, max(xs) + margin_x)
    ax.set_ylim(min(ys) - margin_y, max(ys) + margin_y)
    fig.tight_layout(pad=float(0.02))

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for fmt in formats:
        path = out_dir / f"{name}_structural.{fmt}"
        fig.savefig(path, dpi=int(260), bbox_inches="tight")
        outputs.append(path)
        print(f"Structural layout image: {path}")
    if display_image and Image is not None and display is not None:
        pngs = [p for p in outputs if p.suffix.lower() == ".png"]
        if pngs:
            display(Image(filename=str(pngs[0])))
    plt.close(fig)
    return outputs


def graph_to_networkx(adj):
    import networkx as nx
    H = nx.Graph()
    H.add_nodes_from(sorted(adj))
    for u in sorted(adj):
        for v in sorted(adj[u]):
            if u < v:
                H.add_edge(u, v)
    return H


def draw_planar_layout_image(name, adj, signs, out_dir, description="", formats=("png", "pdf"), display_image=True):
    """Draw a crossing-free planar layout when NetworkX can certify planarity."""
    try:
        import matplotlib.pyplot as plt
        import networkx as nx
    except Exception as exc:
        print(f"Planar layout skipped because NetworkX/Matplotlib is unavailable: {exc}")
        return []

    if len(adj) > MAX_PLANAR_LAYOUT_VERTICES:
        print(f"Planar layout skipped for {name}: {len(adj)} vertices > MAX_PLANAR_LAYOUT_VERTICES={MAX_PLANAR_LAYOUT_VERTICES}.")
        return []

    H = graph_to_networkx(adj)
    is_planar, embedding = nx.check_planarity(H)
    if not is_planar:
        print(f"Planar layout skipped for {name}: NetworkX reports the graph is not planar.")
        return []

    if description:
        display_note(name + " planar layout", description)

    pos = nx.planar_layout(embedding, scale=float(1.0))
    xs = [float(pos[v][0]) for v in H.nodes]
    ys = [float(pos[v][1]) for v in H.nodes]
    width = max(xs) - min(xs) if xs else 1.0
    height = max(ys) - min(ys) if ys else 1.0
    aspect = max(float(0.7), min(float(1.8), width / max(height, 1e-9)))
    fig_w = float(9.5)
    fig_h = float(fig_w / aspect)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_aspect("equal")
    ax.axis("off")

    nx.draw_networkx_edges(H, pos, ax=ax, edge_color="black", width=float(0.75), alpha=float(1.0))
    white_nodes = [v for v in H.nodes if signs[v] == 1]
    black_nodes = [v for v in H.nodes if signs[v] == -1]
    node_size = float(38 if len(adj) > 250 else 58)
    nx.draw_networkx_nodes(H, pos, nodelist=white_nodes, node_color="white", edgecolors="black", linewidths=float(0.8), node_size=node_size, ax=ax)
    nx.draw_networkx_nodes(H, pos, nodelist=black_nodes, node_color="black", edgecolors="black", linewidths=float(0.8), node_size=node_size, ax=ax)
    ax.margins(float(0.05))
    fig.tight_layout(pad=float(0.02))

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for fmt in formats:
        path = out_dir / f"{name}_planar.{fmt}"
        fig.savefig(path, dpi=int(240), bbox_inches="tight")
        outputs.append(path)
        print(f"Planar layout image: {path}")
    if display_image and Image is not None and display is not None:
        pngs = [p for p in outputs if p.suffix.lower() == ".png"]
        if pngs:
            display(Image(filename=str(pngs[0])))
    plt.close(fig)
    return outputs


def _draw_construction_schematic(
    r,
    k,
    h,
    out_dir,
    filename_stem,
    formats=("png", "pdf"),
    size_estimate=None,
    family=False,
    display_image=True,
):
    """Draw the construction exactly in the order used in the proof.

    The proof constructs C_k, completes degrees to S_{r,k}^{(0)}, replaces one
    chosen W-leaf by R_h, and finally applies F_r to every W-vertex of the
    skeleton.  This figure follows those four operations instead of mimicking
    any single informal picture from the PDF.
    """
    try:
        import math
        import matplotlib.pyplot as plt
        from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch
    except Exception as exc:
        print(f"Matplotlib is unavailable, so the construction schematic was skipped: {exc}")
        return []

    X = r**2
    x = r**2 + 1
    y = r + 1
    z = r**2 + r + 1
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(float(15.2), float(8.7)))
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(float(0.0), float(15.2))
    ax.set_ylim(float(0.0), float(8.7))

    segments = []

    def draw_node(px, py, kind="W", radius=0.078, lw=1.2, zorder=7):
        px, py = float(px), float(py)
        face = "white" if kind == "W" else "black"
        circ = Circle((px, py), float(radius), facecolor=face, edgecolor="black", linewidth=float(lw), zorder=zorder)
        ax.add_patch(circ)
        return (px, py)

    def draw_edge(p, q, lw=1.05, style="-", color="black", alpha=1.0, record=True, zorder=3):
        p = (float(p[0]), float(p[1]))
        q = (float(q[0]), float(q[1]))
        ax.plot([p[0], q[0]], [p[1], q[1]], style, color=color, linewidth=float(lw), alpha=float(alpha), zorder=zorder)
        if record and style == "-":
            segments.append((p, q))

    def draw_label(px, py, text, size=8.8, ha="center", va="center", box=True, color="black", zorder=11):
        bbox = dict(facecolor="white", edgecolor="none", alpha=float(0.94), pad=float(1.15)) if box else None
        ax.text(float(px), float(py), text, fontsize=float(size), ha=ha, va=va, bbox=bbox, color=color, zorder=zorder)

    def draw_panel(x0, y0, w0, h0, title, subtitle=None):
        patch = FancyBboxPatch(
            (float(x0), float(y0)),
            float(w0),
            float(h0),
            boxstyle="round,pad=0.10",
            linewidth=float(0.85),
            edgecolor="#7a7a7a",
            facecolor="none",
            linestyle="--",
            zorder=1,
        )
        ax.add_patch(patch)
        draw_label(x0 + 0.20, y0 + h0 - 0.24, title, size=10.0, ha="left", box=True)
        if subtitle:
            draw_label(x0 + 0.20, y0 + h0 - 0.52, subtitle, size=7.6, ha="left", box=True, color="#333333")

    def draw_arrow(x1, y1, x2, y2, text=None):
        arr = FancyArrowPatch(
            (float(x1), float(y1)),
            (float(x2), float(y2)),
            arrowstyle="-|>",
            mutation_scale=float(12),
            linewidth=float(0.9),
            color="#444444",
            zorder=2,
        )
        ax.add_patch(arr)
        if text:
            draw_label((x1 + x2) / 2, (y1 + y2) / 2 + 0.16, text, size=7.2, box=True, color="#333333")

    def sign_u(i):
        return 1 if i % 2 == 0 else -1

    def sign_v(i):
        return -1 if i % 2 == 0 else 1

    def kind_from_sign(sign):
        return "W" if sign == 1 else "B"

    def core_degree(i):
        return 1 + (1 if i > 0 else 0) + (1 if i < k else 0)

    def compact_core_indices():
        if k <= 4:
            return list(range(k + 1)), False
        return [0, 1, k - 1, k], True

    def draw_ladder(x0, y0, width, height, label_nodes=False):
        shown, compressed_core = compact_core_indices()
        if k == 0:
            xs = {0: x0 + 0.50 * width}
        elif compressed_core:
            xs = {shown[0]: x0 + 0.18 * width, shown[1]: x0 + 0.36 * width,
                  shown[2]: x0 + 0.64 * width, shown[3]: x0 + 0.82 * width}
        else:
            step = width / max(1, k)
            xs = {i: x0 + i * step for i in shown}
        top = y0 + height
        bot = y0
        u_pos, v_pos = {}, {}
        for i in shown:
            u_pos[i] = draw_node(xs[i], top, kind_from_sign(sign_u(i)))
            v_pos[i] = draw_node(xs[i], bot, kind_from_sign(sign_v(i)))
            draw_edge(u_pos[i], v_pos[i])
            if label_nodes and (k <= 2 or i in (0, k)):
                draw_label(xs[i], top + 0.18, rf"$u_{i}$", size=7.0, box=False)
                draw_label(xs[i], bot - 0.18, rf"$v_{i}$", size=7.0, box=False)
        for i in shown:
            if i + 1 in u_pos:
                draw_edge(u_pos[i], u_pos[i + 1])
                draw_edge(v_pos[i], v_pos[i + 1])
        if compressed_core:
            draw_label(x0 + 0.50 * width, top, r"$\cdots$", size=13.0)
            draw_label(x0 + 0.50 * width, bot, r"$\cdots$", size=13.0)
        return u_pos, v_pos

    def draw_degree_completion(x0, y0, small=False):
        rnode = 0.066 if small else 0.078
        w = draw_node(x0, y0 + 0.70, "W", radius=rnode)
        p = draw_node(x0 + 0.72, y0 + 0.70, "B", radius=rnode)
        draw_edge(w, p)
        for dy in (0.28, 0.0, -0.28):
            leaf = draw_node(x0 + 1.26, y0 + 0.70 + dy, "W", radius=rnode * 0.86)
            draw_edge(p, leaf, lw=0.92)
        ell = draw_node(x0 + 0.96, y0 + 0.03, "W", radius=rnode * 0.86)
        draw_edge(p, ell, lw=0.92, style="--", color="#555555", record=False)
        draw_label(x0 - 0.03, y0 + 0.84, rf"$w\in W_0$", size=7.5, ha="right")
        draw_label(x0 + 0.38, y0 + 0.40, rf"$X-d_C(w)$ B-copies", size=7.2)
        draw_label(x0 + 1.78, y0 + 0.70, rf"each has $y-1$ W-leaves", size=7.0, ha="left")
        draw_label(x0 + 1.14, y0 + 0.02, rf"chosen leaf $\ell$", size=7.0, ha="left")

        b_y = y0 - 0.30
        b = draw_node(x0 + 0.10, b_y, "B", radius=rnode)
        for dx, dy in [(-0.36, -0.22), (0.02, -0.42), (0.42, -0.22)]:
            leaf = draw_node(x0 + 0.10 + dx, b_y + dy, "W", radius=rnode * 0.86)
            draw_edge(b, leaf, lw=0.92)
        draw_label(x0 + 0.88, b_y, rf"$b\in B_0$: add $y-d_C(b)$ W-leaves", size=7.2, ha="left")
        return p, ell

    def draw_R_branch(root, top_label, current_h=True, compact=False):
        draw_label(root[0] + 0.30, root[1] + 0.18, top_label, size=7.5, ha="left")
        if current_h and h == 0:
            draw_label(root[0], root[1] - 0.38, r"$R_0$: root only", size=7.3)
            return
        child_y = root[1] - (0.62 if compact else 0.70)
        spread = 0.48 if compact else 0.58
        b1 = draw_node(root[0] - spread, child_y, "B", radius=0.066)
        b2 = draw_node(root[0] + spread, child_y, "B", radius=0.066)
        draw_edge(root, b1, lw=0.98)
        draw_edge(root, b2, lw=0.98)
        draw_label(root[0] + 1.02, child_y + 0.10, rf"$X-1$ B-children", size=7.2, ha="left")
        low_y = child_y - (0.58 if compact else 0.66)
        for b in (b1, b2):
            w1 = draw_node(b[0] - 0.18, low_y, "W", radius=0.060)
            w2 = draw_node(b[0] + 0.18, low_y, "W", radius=0.060)
            draw_edge(b, w1, lw=0.88)
            draw_edge(b, w2, lw=0.88)
        draw_label(root[0] + 1.02, low_y, rf"each has $y-1$ roots of $R_{{h-1}}$", size=7.0, ha="left")
        if current_h and h >= 2:
            draw_label(root[0], low_y - 0.40, r"$\vdots$", size=13.0)
            terminal = draw_node(root[0], low_y - 0.74, "W", radius=0.060)
            draw_label(terminal[0] + 0.34, terminal[1], rf"level $h$", size=7.1, ha="left")

    def draw_R_sample(x0, y0, label, depth):
        parent = draw_node(x0, y0, "B", radius=0.060)
        root = draw_node(x0, y0 - 0.43, "W", radius=0.060)
        draw_edge(parent, root, lw=0.86)
        draw_label(x0, y0 + 0.25, label, size=7.3)
        if depth == 0:
            return
        b1 = draw_node(x0 - 0.28, y0 - 0.90, "B", radius=0.054)
        b2 = draw_node(x0 + 0.28, y0 - 0.90, "B", radius=0.054)
        draw_edge(root, b1, lw=0.82)
        draw_edge(root, b2, lw=0.82)
        if depth >= 2:
            draw_label(x0, y0 - 1.18, r"$\vdots$", size=10.0)
        else:
            for b in (b1, b2):
                leaf = draw_node(b[0], y0 - 1.23, "W", radius=0.050)
                draw_edge(b, leaf, lw=0.78)

    def draw_F_rules(x0, y0):
        draw_label(x0 + 1.30, y0 + 2.20, rf"$F_r$: apply to every $W$-vertex of $S_{{r,k,h}}$", size=8.2)
        draw_label(x0 + 0.08, y0 + 1.72, rf"if $d_S(w)=X$", size=7.7, ha="left")
        wA = draw_node(x0 + 0.70, y0 + 1.20, "W", radius=0.064)
        qA = draw_node(x0 + 1.30, y0 + 1.20, "B", radius=0.064)
        draw_edge(wA, qA, lw=0.92)
        for dy in (0.26, 0.0, -0.26):
            leaf = draw_node(x0 + 1.82, y0 + 1.20 + dy, "W", radius=0.056)
            draw_edge(qA, leaf, lw=0.82)
        draw_label(x0 + 2.22, y0 + 1.20, rf"$z-1$ W-leaves", size=7.2, ha="left")

        draw_label(x0 + 0.08, y0 + 0.36, rf"if $d_S(w)=1$", size=7.7, ha="left")
        wB = draw_node(x0 + 0.70, y0 - 0.16, "W", radius=0.064)
        for dx, dy in [(0.58, 0.30), (0.68, 0.0), (0.58, -0.30)]:
            supp = draw_node(x0 + 0.70 + dx, y0 - 0.16 + dy, "B", radius=0.060)
            draw_edge(wB, supp, lw=0.88)
            leaf = draw_node(x0 + 0.70 + dx + 0.36, y0 - 0.16 + dy, "W", radius=0.048)
            draw_edge(supp, leaf, lw=0.72)
        draw_label(x0 + 2.18, y0 - 0.03, rf"$r$ supports, each with $z-1$ leaves", size=7.0, ha="left")
        for dx, dy in [(-0.32, -0.40), (0.06, -0.58)]:
            leaf = draw_node(x0 + 0.70 + dx, y0 - 0.16 + dy, "B", radius=0.054)
            draw_edge(wB, leaf, lw=0.78)
        draw_label(x0 + 1.84, y0 - 0.78, rf"$r(r-1)$ direct B-leaves", size=7.0, ha="left")

    def draw_legend(x, y):
        draw_node(x, y, "W", radius=0.058)
        draw_label(x + 0.22, y, r"$W$ side", size=7.2, ha="left")
        draw_node(x + 1.10, y, "B", radius=0.058)
        draw_label(x + 1.32, y, r"$B$ side", size=7.2, ha="left")

    def crossing_count():
        crossings = 0
        for i, (p1, p2) in enumerate(segments):
            for q1, q2 in segments[i + 1:]:
                shared = {tuple(round(a, 8) for a in p1), tuple(round(a, 8) for a in p2)} & {
                    tuple(round(a, 8) for a in q1), tuple(round(a, 8) for a in q2)
                }
                if shared:
                    continue
                if _proper_segment_crossing(p1, p2, q1, q2):
                    crossings += 1
        return crossings

    if family:
        title = rf"Family $\mathcal{{G}}_{{r,k}}=\{{G_{{r,k,h}}:h\geq 0\}}$ from the construction"
        subtitle = rf"fixed $r={r}$, $k={k}$; $X={X}$, $x={x}$, $y={y}$, $z={z}$; current notebook member $h={h}$"
        note_title = f"Construction schematic for the whole family with r={r}, k={k}"
        note_body = (
            "The picture now follows the proof: build the ladder core, complete degrees to "
            "S^(0), replace one chosen W-leaf by R_h, then apply F_r. Only R_h changes when h varies."
        )
    else:
        title = rf"Construction-compressed drawing of $G_{{{r},{k},{h}}}$"
        size_text = ""
        if size_estimate is not None:
            size_text = rf"; $|V|={size_estimate['G_vertices']:,}$, $|E|={size_estimate['G_edges']:,}$"
        subtitle = rf"$X={X}$, $x={x}$, $y={y}$, $z={z}$, $\omega=k={k}$, $h={h}$" + size_text
        note_title = f"Construction-compressed schematic for G_{{{r},{k},{h}}}"
        note_body = (
            "This is the paper-facing compressed view of the current member. It is organized by "
            "the mathematical construction C_k -> S^(0) -> S_{r,k,h} -> F_r(S_{r,k,h})."
        )
    display_note(note_title, note_body)

    draw_label(7.60, 8.32, title, size=14.0, box=False)
    draw_label(7.60, 7.98, subtitle, size=9.0, box=False)

    draw_panel(0.45, 5.10, 3.35, 2.25, r"1. ladder core $C_k$", rf"$\omega(C_k)=k$")
    draw_ladder(1.05, 5.80, 2.10, 0.70, label_nodes=True)
    draw_label(2.12, 5.40, rf"$W_0=\{{u_i:i even\}}\cup\{{v_i:i odd\}}$", size=6.8)

    draw_panel(4.12, 5.10, 4.15, 2.25, r"2. complete degrees", r"obtain $S_{r,k}^{(0)}$")
    draw_degree_completion(4.76, 5.86, small=True)

    draw_panel(0.45, 0.78, 7.82, 3.85, r"3. replace the chosen leaf", rf"$S_{{r,k,h}}=(S_{{r,k}}^{{(0)}}-\ell)+R_h$")
    parent = draw_node(1.14, 3.72, "B", radius=0.070)
    deleted = draw_node(1.14, 3.08, "W", radius=0.062)
    draw_edge(parent, deleted, style="--", color="#777777", lw=0.95, record=False)
    draw_label(0.78, 3.38, r"delete $\ell$", size=7.4, ha="right")
    root = draw_node(2.12, 3.08, "W", radius=0.072)
    draw_edge(parent, root, lw=1.05)
    draw_arrow(1.36, 3.08, 1.90, 3.08, r"attach root")
    draw_R_branch(root, rf"root of $R_h$", current_h=True, compact=False)
    draw_label(5.35, 3.52, rf"$R_0$ is one $W$-root", size=7.6, ha="left")
    draw_label(5.35, 3.16, rf"$R_h$: root has $X-1$ B-children", size=7.6, ha="left")
    draw_label(5.35, 2.82, rf"each B-child has $y-1$ roots of $R_{{h-1}}$", size=7.6, ha="left")
    if family:
        draw_R_sample(5.02, 1.98, r"$R_0$", 0)
        draw_R_sample(6.12, 1.98, r"$R_1$", 1)
        draw_label(6.92, 1.35, r"$\cdots$", size=11.0)
        draw_R_sample(7.62, 1.98, r"$R_h$", 2)
    else:
        draw_label(5.35, 2.22, rf"current member uses $h={h}$", size=7.9, ha="left")

    draw_panel(8.62, 0.78, 6.10, 6.57, r"4. extend the skeleton", rf"$G_{{r,k,h}}=F_r(S_{{r,k,h}})$")
    draw_F_rules(9.10, 4.45)
    draw_label(9.02, 2.42, r"The extension only attaches trees to $W$-vertices,", size=7.4, ha="left")
    draw_label(9.02, 2.13, r"so it preserves connectedness, simplicity, bipartiteness,", size=7.4, ha="left")
    draw_label(9.02, 1.84, rf"and the cyclomatic number $\omega=k={k}$.", size=7.4, ha="left")
    draw_legend(9.08, 1.22)

    draw_arrow(3.82, 6.20, 4.08, 6.20)
    draw_arrow(8.30, 2.70, 8.58, 2.70)

    crossings = crossing_count()
    print(f"Construction schematic proper edge crossings: {crossings}")

    fig.tight_layout(pad=float(0.03))
    outputs = []
    for fmt in formats:
        path = out_dir / f"{filename_stem}.{fmt}"
        fig.savefig(path, dpi=int(270), bbox_inches="tight")
        outputs.append(path)
        print(f"Construction schematic image: {path}")
    if display_image and display is not None and Image is not None:
        pngs = [p for p in outputs if p.suffix.lower() == ".png"]
        if pngs:
            display(Image(filename=str(pngs[0])))
    plt.close(fig)
    return outputs


def draw_infinite_family_schematic(r, k, h, out_dir, formats=("png", "pdf")):
    """Draw the whole family {G_{r,k,h}: h>=0} according to the construction."""
    return _draw_construction_schematic(
        r,
        k,
        h,
        out_dir,
        filename_stem=f"G_r{r}_k{k}_family_h_variable",
        formats=formats,
        family=True,
        display_image=True,
    )


def draw_compressed_member_schematic(r, k, h, size_estimate, out_dir, formats=("png", "pdf"), display_image=True):
    """Draw the current member G_{r,k,h} as the proof construction, compressed."""
    return _draw_construction_schematic(
        r,
        k,
        h,
        out_dir,
        filename_stem=f"G_r{r}_k{k}_h{h}_compressed_schematic",
        formats=formats,
        size_estimate=size_estimate,
        family=False,
        display_image=display_image,
    )


print("Definitions cell executed: graph constructors and visualization helpers are loaded.")


In [4]:
size_estimate = estimate_G_r_k_h_size(r, k, h)
print("Preflight size estimate:")
print(f"  |V(S_{{r,k,h}})| = {size_estimate['S_vertices']:,}, |E(S_{{r,k,h}})| = {size_estimate['S_edges']:,}")
print(f"  |V(G_{{r,k,h}})| = {size_estimate['G_vertices']:,}, |E(G_{{r,k,h}})| = {size_estimate['G_edges']:,}")
if MAX_BUILD_VERTICES is not None and size_estimate['G_vertices'] > MAX_BUILD_VERTICES:
    raise RuntimeError(
        f"This parameter choice would construct {size_estimate['G_vertices']:,} vertices, "
        f"above MAX_BUILD_VERTICES={MAX_BUILD_VERTICES:,}. "
        "Use a smaller h, or set MAX_BUILD_VERTICES=None if you intentionally want to try it. "
        "For the infinite-family picture, you do not need a large h; h=0,1,2 is enough."
    )
if SHOW_FULL_MATRICES and size_estimate['G_vertices'] > MAX_FULL_MATRIX_DISPLAY_VERTICES:
    print(
        f"SHOW_FULL_MATRICES=True but |V(G)|={size_estimate['G_vertices']:,} is large; "
        "turning full matrix display off for this run."
    )
    SHOW_FULL_MATRICES = False

G, adj, signs, kinds, S_adj, S_signs, S_kinds = build_G_r_k_h(r, k, h)
Q, residual, q_sigma, recurrence = verify_basic_identities(G, signs, r)
W4 = walk_matrix_first_four(Q)
rank_W4 = W4.rank()

a = r**2 + r + 3
b = r + 2
t = r
omega = G.num_edges() - G.num_verts() + 1
plus_count = sum(1 for v in signs if signs[v] == 1)
minus_count = sum(1 for v in signs if signs[v] == -1)

print(f"Parameters: r={r}, k={k}, h={h}")
print(f"(a,b,t)=({a},{b},{t})")
print(f"|V(G)|={G.num_verts()}, |E(G)|={G.num_edges()}, omega=|E|-|V|+1={omega}")
print(f"Connected: {G.is_connected()}, Bipartite: {G.is_bipartite()}, Simple: {not G.allows_multiple_edges() and not G.allows_loops()}")
zero_main_witness = plus_count - minus_count
qmain_discriminant = a*a - 4*b
qmain_polynomial = f"lambda*(lambda^2 - {a}*lambda + {b})"
qmain_values_exact = ["0", f"({a} - sqrt({qmain_discriminant}))/2", f"({a} + sqrt({qmain_discriminant}))/2"]

print(f"Bipartition signs: |V_+|={plus_count}, |V_-|={minus_count}, b*(|V_+|-|V_-|)={b*zero_main_witness}, t*|V|={t*G.num_verts()}")
print(f"rank([1, Q1, Q^2 1, Q^3 1]) = {rank_W4}")
print(f"max |Q^2*1 - a*Q*1 + b*1 - t*sigma| = {max(abs(x) for x in residual)}")
print(f"max |Q*sigma| = {max(abs(x) for x in q_sigma)}")
print(f"max |Q^3*1 - a*Q^2*1 + b*Q*1| = {max(abs(x) for x in recurrence)}")
print()
print("Q-main eigenvalue verification:")
print(f"  number of Q-main eigenvalues = {rank_W4}")
print(f"  relative minimal polynomial of Q with respect to 1: {qmain_polynomial}")
print(f"  Q-main eigenvalues: {', '.join(qmain_values_exact)}")
print(f"  zero eigenvalue is Q-main because <1,sigma> = |V_+|-|V_-| = {zero_main_witness} != 0")

display_note(
    "Q-main eigenvalue result",
    f"The walk space generated by `1` has dimension `{rank_W4}`. The recurrence `Q^3 1 - {a} Q^2 1 + {b} Q1 = 0` gives the relative polynomial `{qmain_polynomial}`, so this run verifies exactly three Q-main eigenvalues: `{', '.join(qmain_values_exact)}`. The vector `sigma` satisfies `Q sigma = 0`, and `<1,sigma> = {zero_main_witness}`, hence `0` is one of the Q-main eigenvalues."
)

if omega != k:
    raise AssertionError(f"Expected omega={k}, got {omega}")
if rank_W4 != 3:
    raise AssertionError(f"Expected Q-main eigenvalue count 3, got {rank_W4}")
if zero_main_witness == 0:
    raise AssertionError("0 is a Q-eigenvalue, but it is not Q-main because <1,sigma>=0")
if any(residual):
    raise AssertionError("The admissible equation failed")
if any(q_sigma):
    raise AssertionError("Q*sigma is not zero")
if any(recurrence):
    raise AssertionError("The Q-walk recurrence failed")

print("All checks passed.")

Preflight size estimate:
  |V(S_{r,k,h})| = 13, |E(S_{r,k,h})| = 12
  |V(G_{r,k,h})| = 148, |E(G_{r,k,h})| = 147


Parameters: r=2, k=0, h=0
(a,b,t)=(9,4,2)
|V(G)|=148, |E(G)|=147, omega=|E|-|V|+1=0
Connected: True, Bipartite: True, Simple: True
Bipartition signs: |V_+|=111, |V_-|=37, b*(|V_+|-|V_-|)=296, t*|V|=296
rank([1, Q1, Q^2 1, Q^3 1]) = 3
max |Q^2*1 - a*Q*1 + b*1 - t*sigma| = 0
max |Q*sigma| = 0
max |Q^3*1 - a*Q^2*1 + b*Q*1| = 0

Q-main eigenvalue verification:
  number of Q-main eigenvalues = 3
  relative minimal polynomial of Q with respect to 1: lambda*(lambda^2 - 9*lambda + 4)
  Q-main eigenvalues: 0, (9 - sqrt(65))/2, (9 + sqrt(65))/2
  zero eigenvalue is Q-main because <1,sigma> = |V_+|-|V_-| = 74 != 0


### Q-main eigenvalue result
The walk space generated by `1` has dimension `3`. The recurrence `Q^3 1 - 9 Q^2 1 + 4 Q1 = 0` gives the relative polynomial `lambda*(lambda^2 - 9*lambda + 4)`, so this run verifies exactly three Q-main eigenvalues: `0, (9 - sqrt(65))/2, (9 + sqrt(65))/2`. The vector `sigma` satisfies `Q sigma = 0`, and `<1,sigma> = 74`, hence `0` is one of the Q-main eigenvalues.

All checks passed.


In [5]:
display_note(
    "Matrix output",
    "This cell reports the signless Laplacian matrix `Q=A+D` and the first four columns of the Q-walk matrix. For large graphs the matrices are computed but not displayed in full, because a full matrix dump can make VSCode notebooks slow."
)

print("Signless Laplacian matrix Q = A + D")
print(f"Q has shape {Q.nrows()} x {Q.ncols()}")
if SHOW_FULL_MATRICES:
    show(Q)
else:
    print("Full Q display skipped because SHOW_FULL_MATRICES=False.")

print("First four columns of the Q-walk matrix [1, Q1, Q^2 1, Q^3 1]")
print(f"W4 has shape {W4.nrows()} x {W4.ncols()}")
if SHOW_FULL_MATRICES:
    show(W4)
else:
    print("Full W4 display skipped because SHOW_FULL_MATRICES=False.")

print(f"rank(W4) = {rank_W4}")


if EXPORT_MATRIX_FILES:
    display_note(
        "Matrix files exported",
        "For large graphs the notebook stores Q and the Q-walk matrix on disk instead of printing huge matrices into VSCode. Q is saved sparsely, and W4 is saved as a four-column table keyed by the vertex order."
    )
    matrix_metadata = {
        "r": r,
        "k": k,
        "h": h,
        "vertices": G.num_verts(),
        "edges": G.num_edges(),
        "omega": omega,
        "rank_W4": int(rank_W4),
        "Q_main_eigenvalue_count": int(rank_W4),
        "relative_minimal_polynomial": qmain_polynomial,
        "Q_main_eigenvalues": qmain_values_exact,
        "zero_is_Q_main": bool(zero_main_witness != 0),
    }
    matrix_dir = Path("matrices") / f"G_r{r}_k{k}_h{h}"
    matrix_paths = export_matrix_files(Q, W4, G.vertices(sort=True), matrix_dir, matrix_metadata)


### Matrix output
This cell reports the signless Laplacian matrix `Q=A+D` and the first four columns of the Q-walk matrix. For large graphs the matrices are computed but not displayed in full, because a full matrix dump can make VSCode notebooks slow.

Signless Laplacian matrix Q = A + D
Q has shape 148 x 148
Full Q display skipped because SHOW_FULL_MATRICES=False.
First four columns of the Q-walk matrix [1, Q1, Q^2 1, Q^3 1]
W4 has shape 148 x 4
Full W4 display skipped because SHOW_FULL_MATRICES=False.
rank(W4) = 3


### Matrix files exported
For large graphs the notebook stores Q and the Q-walk matrix on disk instead of printing huge matrices into VSCode. Q is saved sparsely, and W4 is saved as a four-column table keyed by the vertex order.

Matrix artifact directory: /Users/chahangxi/Desktop/others/sagemath/Q_main_G_r_k_h_verifier_project/matrices/G_r2_k0_h0
  Q_sparse_entries_csv: matrices/G_r2_k0_h0/Q_sparse_entries.csv
  W4_csv: matrices/G_r2_k0_h0/Q_walk_first_four_columns.csv
  vertex_order_csv: matrices/G_r2_k0_h0/vertex_order.csv
  Q_mtx: matrices/G_r2_k0_h0/Q_signless_laplacian.mtx
  metadata_json: matrices/G_r2_k0_h0/matrix_metadata.json
  readme_md: matrices/G_r2_k0_h0/README_matrices.md


In [ ]:
# Visualization and large-graph exports.
# Practical drawing choices used here:
# - The main notebook preview is a construction-aware structural layout.
# - The ladder core is drawn as a ladder; attached trees are expanded outward in disjoint intervals.
# - This follows the tidy tree / layered drawing style used in graph-theory figures.
# - Graphviz/Gephi exports are still produced for interactive exploration.

viz_dir = Path("figures") / f"G_r{r}_k{k}_h{h}"
viz_dir.mkdir(parents=True, exist_ok=True)

display_note(
    "Visualization output directory",
    f"All figure/export files for this parameter choice are written to `{viz_dir.resolve()}`. The project keeps figures under `figures/G_r*_k*_h*/` and matrices under `matrices/G_r*_k*_h*/`. The notebook displays compressed paper-style previews for large graphs; exact DOT/GEXF files remain available for Graphviz or Gephi exploration."
)

paper_vertex_colors = {
    "white": [v for v in G.vertices(sort=True) if signs[v] == 1],
    "black": [v for v in G.vertices(sort=True) if signs[v] == -1],
}

# 1. Paper-style compressed schematic for the current member G_{r,k,h}.
if DRAW_COMPRESSED_MEMBER_SCHEMATIC:
    draw_compressed_member_schematic(
        r=r,
        k=k,
        h=h,
        size_estimate=size_estimate,
        out_dir=viz_dir,
        formats=COMPRESSED_MEMBER_SCHEMATIC_FORMATS,
        display_image=DISPLAY_STRUCTURAL_IMAGE_IN_NOTEBOOK,
    )

# 2. Optional direct Sage drawing for small debugging pictures.
if DRAW_DIRECT_SAGE_PLOT:
    display_note(
        "Optional exact Sage drawing",
        "This is a quick exact drawing of every vertex and edge of G_{r,k,h}. It omits vertex numbers. For paper use, the structural layout below is usually clearer."
    )
    if G.num_verts() <= MAX_SAGE_DRAW_VERTICES:
        vertex_size = 90 if G.num_verts() <= 80 else 45
        show(G.plot(layout="spring", vertex_colors=paper_vertex_colors, vertex_size=vertex_size, vertex_labels=False, edge_thickness=1.0, figsize=9))
        if SAVE_GRAPH_IMAGE:
            G.plot(layout="spring", vertex_colors=paper_vertex_colors, vertex_size=45, vertex_labels=False, edge_thickness=1.0, figsize=9).save(GRAPH_IMAGE_PATH)
            print(f"Saved Sage plot image to {GRAPH_IMAGE_PATH}")
    else:
        print(f"Skipped direct Sage plot: {G.num_verts()} vertices > MAX_SAGE_DRAW_VERTICES={MAX_SAGE_DRAW_VERTICES}.")
else:
    print("Direct Sage spring plot is disabled by default. Set DRAW_DIRECT_SAGE_PLOT=True only for small debugging pictures.")

# 3. Structure-aware exact full graph drawing.  This is useful for small graphs,
# but a literal drawing of thousands of vertices is no longer a readable paper figure.
if DRAW_STRUCTURAL_LAYOUT:
    if G.num_verts() <= MAX_STRUCTURAL_LAYOUT_VERTICES:
        draw_structural_layout_image(
            name=f"G_r{r}_k{k}_h{h}_full",
            adj=adj,
            signs=signs,
            k=k,
            out_dir=viz_dir,
            description=(
                f"Exact graph `G_{{{r},{k},{h}}}` with all `{G.num_verts()}` vertices and `{G.num_edges()}` edges. "
                "The ladder core is drawn explicitly and every attached tree is expanded outward in its own interval. "
                "The printed crossing count checks proper straight-line edge crossings."
            ),
            formats=STRUCTURAL_LAYOUT_FORMATS,
            display_image=DISPLAY_STRUCTURAL_IMAGE_IN_NOTEBOOK,
            kinds=kinds,
        )
    else:
        display_note(
            "Exact full graph drawing skipped",
            f"The exact graph has `{G.num_verts()}` vertices, above `MAX_STRUCTURAL_LAYOUT_VERTICES={MAX_STRUCTURAL_LAYOUT_VERTICES}`. "
            "The compressed schematic above is the intended paper-style view; exact vertices and edges are still exported as matrix, DOT, and GEXF files."
        )

# 4. Optional generic planar layout. It is crossing-free but often less readable.
if DRAW_PLANAR_LAYOUT:
    draw_planar_layout_image(
        name=f"G_r{r}_k{k}_h{h}_full",
        adj=adj,
        signs=signs,
        out_dir=viz_dir,
        description=(
            f"Generic planar embedding of `G_{{{r},{k},{h}}}`. This is kept as a fallback/diagnostic; "
            "for reading the construction, prefer the structural layout above."
        ),
        formats=PLANAR_LAYOUT_FORMATS,
        display_image=DISPLAY_PLANAR_IMAGE_IN_NOTEBOOK,
    )

# 5. Exact full graph export for Graphviz/Gephi exploration.
export_visualization_bundle(
    name=f"G_r{r}_k{k}_h{h}_full",
    adj=adj,
    signs=signs,
    kinds=kinds,
    out_dir=viz_dir,
    description=(
        f"Exploratory export of the exact full graph `G_{{{r},{k},{h}}}`. Graphviz force-directed layouts can be useful for exploration but are not guaranteed to be crossing-free; the structural PNG above is the paper-style preview."
    ),
    engine=GRAPHVIZ_FULL_ENGINE,
    render_vertex_limit=MAX_GRAPHVIZ_RENDER_VERTICES,
    label_vertices=False,
    node_width=0.055 if G.num_verts() > 150 else 0.075,
    edge_width=0.55 if G.num_verts() > 150 else 0.75,
    display_svg=DISPLAY_SVG_IN_NOTEBOOK,
)
if not USE_GRAPHVIZ_SFDP:
    print("Graphviz SVG rendering is disabled by default to avoid long sfdp runs on large exact graphs. DOT/GEXF files were still exported.")

# 6. Skeleton view. This is often the readable structural object behind the construction.
if DRAW_SKELETON_VIEW:
    S_graph = graph_from_adj(S_adj)
    display_note(
        f"Skeleton S_{{{r},{k},{h}}}",
        f"This suppresses the final local expansion `F_r` and shows the k-cyclic skeleton used to build the graph. "
        f"Here `|V(S)|={S_graph.num_verts()}`, `|E(S)|={S_graph.num_edges()}`, and `omega(S)={S_graph.num_edges()-S_graph.num_verts()+1}`."
    )
    if DRAW_STRUCTURAL_LAYOUT:
        draw_structural_layout_image(
            name=f"G_r{r}_k{k}_h{h}_skeleton",
            adj=S_adj,
            signs=S_signs,
            k=k,
            out_dir=viz_dir,
            description="Structure-aware drawing of the skeleton. This is the compact view for seeing the ladder core and the variable branch R_h.",
            formats=STRUCTURAL_LAYOUT_FORMATS,
            display_image=DISPLAY_STRUCTURAL_IMAGE_IN_NOTEBOOK,
        )
    if DRAW_PLANAR_LAYOUT:
        draw_planar_layout_image(
            name=f"G_r{r}_k{k}_h{h}_skeleton",
            adj=S_adj,
            signs=S_signs,
            out_dir=viz_dir,
            description="Generic planar drawing of the skeleton, kept as a fallback diagnostic.",
            formats=PLANAR_LAYOUT_FORMATS,
            display_image=DISPLAY_PLANAR_IMAGE_IN_NOTEBOOK,
        )
    export_visualization_bundle(
        name=f"G_r{r}_k{k}_h{h}_skeleton",
        adj=S_adj,
        signs=S_signs,
        kinds=S_kinds,
        out_dir=viz_dir,
        description="Graphviz/Gephi export of the skeleton for interactive exploration.",
        engine=GRAPHVIZ_SKELETON_ENGINE,
        render_vertex_limit=MAX_GRAPHVIZ_RENDER_VERTICES,
        label_vertices=False,
        node_width=0.095,
        edge_width=0.95,
        display_svg=DISPLAY_SVG_IN_NOTEBOOK,
    )

# 7. Paper-style schematic for the whole infinite family with r,k fixed and h varying.
if DRAW_INFINITE_FAMILY_SCHEMATIC:
    display_note(
        f"Paper-style infinite-family schematic for fixed r={r}, k={k}",
        "This is not a force-directed plot of one large graph. It is a compressed graph-theory figure for the whole family as h varies: representative branches are drawn once, and multiplicities are written as labels. This is the figure to use when explaining why fixed r,k give infinitely many k-cyclic graphs."
    )
    draw_infinite_family_schematic(r, k, h, viz_dir, formats=FAMILY_SCHEMATIC_FORMATS)
